
# Dataset Exploration
This notebook inspects the converted dataset, calculates distributions (image sizes, class distributions, box dimensions), and identifies missing or malformed data.
Outputs are saved to `outputs/metrics/`.


In [ ]:

import os
import yaml
import json
from pathlib import Path
import cv2
from tqdm.auto import tqdm
import sys

# Ensure src is in path
sys.path.append(str(Path.cwd().parent))

from src.aeronetra.visualization.plots import plot_class_distribution, plot_size_distribution, plot_objects_per_image


In [ ]:

# Configuration via environment variables or defaults
dataset_dir_env = os.getenv("DATASET_DIR")
if not dataset_dir_env:
    print("Warning: DATASET_DIR environment variable not set. Using fallback relative path '../data'.")
    dataset_base = Path("../data")
else:
    dataset_base = Path(dataset_dir_env)

# Example using the YOLO converted dataset
config_path = Path("../configs/datasets/visdrone_yolo.yaml")

if not config_path.exists():
    raise FileNotFoundError(f"Configuration file not found: {config_path}")

with open(config_path, "r") as f:
    config = yaml.safe_load(f)

# Resolve paths by replacing placeholder
train_img_dir = Path(config["paths"]["train"]["images"].replace("${DATASET_DIR}", str(dataset_base)))
train_lbl_dir = Path(config["paths"]["train"]["labels"].replace("${DATASET_DIR}", str(dataset_base)))
class_names = {int(k): v for k, v in config["classes"].items()}

out_metrics_dir = Path("../outputs/metrics")
out_metrics_dir.mkdir(parents=True, exist_ok=True)


In [ ]:

if not train_img_dir.exists() or not any(train_img_dir.iterdir()):
    print(f"Error: Dataset directory {train_img_dir} does not exist or is empty.")
    print("Please configure DATASET_DIR and ensure you have run the conversion script.")
    # We will stop execution cleanly if data is absent
else:
    print(f"Found images directory: {train_img_dir}")
    print(f"Found labels directory: {train_lbl_dir}")


In [ ]:

def analyze_dataset(images_dir: Path, labels_dir: Path):
    if not images_dir.exists():
        return None
        
    stats = {
        "num_images": 0,
        "num_annotations": 0,
        "missing_labels": 0,
        "empty_labels": 0,
        "out_of_bounds": 0
    }
    
    class_counts = {c: 0 for c in class_names.keys()}
    widths = []
    heights = []
    areas = []
    objects_per_image = []
    
    for img_path in tqdm(list(images_dir.glob("*.jpg")), desc="Analyzing images"):
        stats["num_images"] += 1
        lbl_path = labels_dir / f"{img_path.stem}.txt"
        
        if not lbl_path.exists():
            stats["missing_labels"] += 1
            continue
            
        img = cv2.imread(str(img_path))
        if img is None:
            continue
        img_h, img_w = img.shape[:2]
        
        with open(lbl_path, "r") as f:
            lines = f.readlines()
            
        if not lines:
            stats["empty_labels"] += 1
            objects_per_image.append(0)
            continue
            
        objects_per_image.append(len(lines))
        for line in lines:
            parts = line.strip().split()
            if len(parts) != 5:
                continue
            stats["num_annotations"] += 1
            
            c = int(parts[0])
            if c in class_counts:
                class_counts[c] += 1
                
            w = float(parts[3]) * img_w
            h = float(parts[4]) * img_h
            
            widths.append(w)
            heights.append(h)
            areas.append(w * h)
            
            x_c = float(parts[1])
            y_c = float(parts[2])
            if x_c - float(parts[3])/2 < 0 or y_c - float(parts[4])/2 < 0 or x_c + float(parts[3])/2 > 1 or y_c + float(parts[4])/2 > 1:
                stats["out_of_bounds"] += 1
                
    return stats, class_counts, widths, heights, areas, objects_per_image

if train_img_dir.exists():
    results = analyze_dataset(train_img_dir, train_lbl_dir)


In [ ]:

if train_img_dir.exists() and results:
    stats, class_counts, widths, heights, areas, objects_per_image = results
    
    # Save Report
    report = {
        "dataset": config.get("dataset_name", "VisDrone"),
        "statistics": stats,
        "class_counts": class_counts
    }
    with open(out_metrics_dir / "dataset_audit_report.json", "w") as f:
        json.dump(report, f, indent=2)
        
    print("Saved audit report to outputs/metrics/dataset_audit_report.json")
    
    # Plotting
    plot_class_distribution(class_counts, class_names, out_metrics_dir / "class_dist.png")
    plot_size_distribution(areas, "Bounding Box Area Distribution", "Area (pixels^2)", out_metrics_dir / "area_dist.png")
    plot_size_distribution(widths, "Bounding Box Width Distribution", "Width (pixels)", out_metrics_dir / "width_dist.png")
    plot_objects_per_image(objects_per_image, out_metrics_dir / "objects_per_image.png")
    print("Saved plots to outputs/metrics/")
